# Following the W&B hyperparameter sweeps tutorial with Bayesian optimization

I worked through the W&B hyperparameter sweeps tutorial and tried setting up a sweep with Bayesian optimization on the Iris dataset. Here's what got it working and where I got stuck.

last_verified: 2026-07-10 · W&B n/a

## Setup

I started with the Iris dataset since it's small and loads quickly. I want to sweep over RandomForest hyperparameters and let W&B pick the next config using Bayesian optimization.

In [ ]:
import wandb
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

# Load data once so each sweep run uses the same split
X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

## Define the training function

W&B calls this once per sweep run. I initialize a run, build a model from the sweep config, log metrics, and let W&B track everything.

In [ ]:
def train():
    """Training function invoked by the W&B sweep agent."""
    with wandb.init() as run:
        config = run.config

        model = RandomForestClassifier(
            n_estimators=config.n_estimators,
            max_depth=config.max_depth,
            random_state=42,
        )
        model.fit(X_train, y_train)
        preds = model.predict(X_test)

        wandb.log({"accuracy": accuracy_score(y_test, preds)})

        # not sure why this works yet — just logging accuracy for now
        # TODO: add precision/recall and model artifact logging.

## Configure the sweep with Bayesian optimization

The key part is `method: bayes`. W&B uses this to pick the next hyperparameter config based on past results instead of trying every combination blindly.

In [ ]:
sweep_config = {
    "method": "bayes",
    "metric": {"name": "accuracy", "goal": "maximize"},
    "parameters": {
        "n_estimators": {"values": [50, 100, 200]},
        "max_depth": {"values": [3, 5, 7]},
    },
}

sweep_id = wandb.sweep(sweep_config, project="iris-bayes-sweep")
print(f"Sweep ID: {sweep_id}")

## Run the sweep

Launch the agent with a count so it stops after a fixed number of runs. Without a count it keeps going until you kill it.

In [ ]:
# Run 5 iterations of the Bayesian sweep
wandb.agent(sweep_id, function=train, count=5, project="iris-bayes-sweep")

# After the sweep finishes, I can look at the best run in the W&B UI
# or pull it back programmatically.

## What tripped me up

1. **Forgot to call `wandb.init()` inside `train`** — I wrote the function without the context manager and W&B complained that no run was active. The `with wandb.init()` wrapper is mandatory inside the sweep callback.
2. **No metric goal set** — I left out `goal` in the metric block and W&B defaulted to minimizing, which is backwards for accuracy. Adding `"goal": "maximize"` fixed it.
3. **Running without a project name** — `wandb.sweep()` needs a project so the UI groups the runs. I passed `project="iris-bayes-sweep"` to both `sweep` and `agent`.
4. **Agent runs forever without `count`** — I launched `wandb.agent(sweep_id, function=train)` with no count and it kept spawning runs. Adding `count=5` gives me a finite experiment I can inspect.

In [ ]:
# Pull the best run back from the W&B API
api = wandb.Api()
sweep = api.sweep("my-entity/iris-bayes-sweep/{sweep_id}")
best_run = max(
    sweep.runs,
    key=lambda r: r.summary.get("accuracy", 0),
)
print(f"Best run: {best_run.id} — accuracy: {best_run.summary['accuracy']:.4f}")
print(f"Config: {dict(best_run.config)}")

## What I'd try next

I want to add more metrics (precision, recall) and log the trained model as an artifact so I can compare the best model directly in the registry. I'd also like to try a wider parameter grid and see how the Bayesian search converges compared to a random search baseline.